# Vertex Vision Results Analysis

Analysis-only notebook. It reads the append-only JSONL ledger and never calls Vertex AI. By default it reports the most recently recorded experiment. Set `EXPERIMENT_KEY` to compare or reproduce a specific run.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
from IPython.display import display

def find_backend(start: Path) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        if directory.name == 'backend' and (directory / 'experiments').is_dir():
            return directory
        candidate = directory / 'backend'
        if (candidate / 'experiments').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not locate the backend directory from {start.resolve()}')

BACKEND = find_backend(Path.cwd())
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from evals import description_scoring, field_scoring, gold, normalization, offline_normalized_run, skills_text_scoring, skills_hybrid_scoring
gold = importlib.reload(gold)  # Always reload reviewed gold before offline scoring.
description_scoring = importlib.reload(description_scoring)
skills_text_scoring = importlib.reload(skills_text_scoring)
skills_hybrid_scoring = importlib.reload(skills_hybrid_scoring)
normalization = importlib.reload(normalization)
field_scoring = importlib.reload(field_scoring)
offline_normalized_run = importlib.reload(offline_normalized_run)
NORMALIZATION_RULES = normalization.NORMALIZATION_RULES
run_normalized_scoring = offline_normalized_run.run_normalized_scoring
LEDGER_PATH = BACKEND / 'experiments' / 'runs' / 'vertex_vision_raw_baseline' / 'attempts.jsonl'
EXPORT_DIR = LEDGER_PATH.parent / 'analysis_exports'
EXPERIMENT_KEY = None  # None selects the normalized experiment created below
CREATE_NORMALIZED_EXPERIMENT = True
NORMALIZATION_SOURCE_EXPERIMENT_KEY = 'gemma-4-26b-a4b-it-maas:v016_gemma_v010_comparable:e732b80515f0285e3cd2e3f6efbb5f4ea3bb8dc1a70480fe38a2c60b7fcac2fc:temp=0.0:thinking=medium:dpi=150:evaluation=gemma4_26b_same_prompt_as_best_gemini_v010_v016:normalize_for_scoring=False:location_policy=locality_only_v001:provider=815ed00b074fd8badd310565a5be60d178ab3e174e760ebfae1405a2a3099151:gold=education_source_review_v001_source_faithful_skills_v002_project_certification_source_review_v001_description_ocr_review_v001_certification_dedicated_sections_v001_skill_sections_v001_corrected_profiles_v001_reviewed_skill_certification_overlays_v003'
PROMPT_DISPLAY_ALIASES = {}
TARGET_F1 = 0.75       # Milestone 3/5 documented minimum
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Ledger:', LEDGER_PATH)

## Load the selected experiment

In [ ]:
def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f'Run the evaluation notebook first: {path}')
    rows = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f'Corrupt ledger line {line_number}: {exc}') from exc
    return rows

if CREATE_NORMALIZED_EXPERIMENT:
    display(pd.DataFrame({'normalization_rule': NORMALIZATION_RULES}))
    normalized_run = run_normalized_scoring(
        source_ledger=LEDGER_PATH, output_ledger=LEDGER_PATH,
        source_experiment_key=NORMALIZATION_SOURCE_EXPERIMENT_KEY,
    )
    display(pd.DataFrame([normalized_run]))

ledger = pd.DataFrame(read_jsonl(LEDGER_PATH))
# Preserve immutable ledger values, but use the true prompt identity in tables/charts.
if 'prompt_version' in ledger.columns:
    ledger['prompt_version'] = ledger['prompt_version'].replace(PROMPT_DISPLAY_ALIASES)
success_log = ledger[ledger.status.eq('success')].copy()
if success_log.empty:
    raise ValueError('The ledger contains no successful resume evaluations.')
selected_key = (EXPERIMENT_KEY or
                (normalized_run['evaluation_key'] if CREATE_NORMALIZED_EXPERIMENT else None) or
                success_log.sort_values('timestamp_utc').iloc[-1].experiment_key)
selected = (success_log[success_log.experiment_key.eq(selected_key)]
            .sort_values('timestamp_utc').drop_duplicates('resume_id', keep='last'))
if selected.empty:
    raise ValueError(f'No successful rows for experiment: {selected_key}')
failures = ledger[ledger.experiment_key.eq(selected_key) & ledger.status.eq('failure')]
metadata = selected.iloc[-1]
display(pd.DataFrame([{
    'experiment_key': selected_key, 'model': metadata.get('model'),
    'prompt_version': metadata.get('prompt_version'), 'resumes': selected.resume_id.nunique(),
    'failed_attempts': len(failures), 'value_normalization': metadata.get('value_normalization'),
}]))

## Build field-level evidence

In [ ]:
rows = []
for record in selected.to_dict('records'):
    for result in record.get('field_counts', []):
        rows.append({
            'resume_id': record['resume_id'], 'split': record.get('split'),
            'category': record.get('category'), 'resume_total_seconds': record.get('resume_total_seconds'),
            **result,
        })
counts = pd.DataFrame(rows)
if counts.empty:
    raise ValueError('Successful rows do not contain field_counts.')

def divide(n, d):
    return n / d if d else np.nan

def summarize(frame):
    totals = frame.groupby('field', as_index=False)[
        ['tp', 'fp', 'fn', 'presence_tp', 'presence_fp', 'presence_fn', 'presence_tn',
         'predicted_count', 'gold_count']
    ].sum()
    totals['precision'] = [divide(tp, tp + fp) for tp, fp in zip(totals.tp, totals.fp)]
    totals['recall'] = [divide(tp, tp + fn) for tp, fn in zip(totals.tp, totals.fn)]
    totals['f1'] = [divide(2*tp, 2*tp + fp + fn) for tp, fp, fn in zip(totals.tp, totals.fp, totals.fn)]
    totals['presence_accuracy'] = [
        divide(tp + tn, tp + tn + fp + fn)
        for tp, tn, fp, fn in zip(totals.presence_tp, totals.presence_tn, totals.presence_fp, totals.presence_fn)
    ]
    totals['field_present_resumes'] = totals.presence_tp + totals.presence_fn
    totals['field_absent_resumes'] = totals.presence_fp + totals.presence_tn
    return totals

summary = summarize(counts)

def show_section(title, fields):
    section = summary[summary.field.isin(fields)].set_index('field').reindex(fields).reset_index()
    display(section[['field', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1',
                     'field_present_resumes', 'field_absent_resumes', 'presence_tn',
                     'presence_accuracy']].round(4))
    chart = section.melt(id_vars='field', value_vars=['precision', 'recall', 'f1'],
                         var_name='metric', value_name='score')
    figure = px.bar(
        chart, x='field', y='score', color='metric', barmode='group',
        range_y=[0, 1], title=f'{title}: latest selected experiment',
        hover_data={'score': ':.4f'},
    )
    figure.add_hline(
        y=TARGET_F1, line_dash='dash', line_color='red', line_width=2,
        annotation_text=f'Target F1 = {TARGET_F1:.0%}',
        annotation_position='top left',
    )
    figure.update_layout(xaxis_title='Field', yaxis_title='Exact-match score')
    figure.show()
    errors = counts[counts.field.isin(fields) & (counts.fp.gt(0) | counts.fn.gt(0))]
    display(errors[['resume_id', 'field', 'tp_values', 'fp_values', 'fn_values',
                    'predicted_values', 'gold_values']].sort_values(['field', 'resume_id']))
    return section, errors

## Raw schema adherence

In [ ]:
schema_rows = selected[['resume_id', 'raw_schema_valid', 'raw_schema_errors']].copy()
schema_valid_mask = schema_rows['raw_schema_valid'].fillna(False).astype(bool)
schema_rows['schema_status'] = np.where(
    schema_valid_mask,
    '✅ ENFORCED — no manual schema review needed',
    '❌ FAILED — inspect schema errors',
)
display(schema_rows[['resume_id', 'schema_status']])
invalid_schema = schema_rows.loc[~schema_valid_mask]
if not invalid_schema.empty:
    print('Schema failures requiring review:')
    display(invalid_schema[['resume_id', 'raw_schema_errors']])
print({'schema_valid_resumes': int(schema_valid_mask.sum()),
       'schema_invalid_resumes': int((~schema_valid_mask).sum())})

## Section score overview

This ranks the six extraction sections using micro-aggregated exact-match counts from the latest successful result per resume in the selected experiment. The lowest F1 section is the first place to investigate.

In [ ]:
SECTION_FIELDS = {
    'Contact': ['contact.name', 'contact.location'],
    'Skills': ['skills'],
    'Education': ['education.degree', 'education.field', 'education.institution',
                  'education.start_year', 'education.end_year'],
    'Experience': ['experience.job_title', 'experience.company', 'experience.location',
                   'experience.start_date', 'experience.end_date', 'experience.current_role'],
    'Projects': ['projects.name', 'projects.technologies'],
    'Certifications': ['certifications.name', 'certifications.issuer', 'certifications.year'],
}

section_rows = []
for section_name, section_fields in SECTION_FIELDS.items():
    section_counts = counts[counts.field.isin(section_fields)]
    tp = int(section_counts.tp.sum())
    fp = int(section_counts.fp.sum())
    fn = int(section_counts.fn.sum())
    section_rows.append({
        'section': section_name,
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': divide(tp, tp + fp),
        'recall': divide(tp, tp + fn),
        'f1': divide(2 * tp, 2 * tp + fp + fn),
    })

section_summary = (pd.DataFrame(section_rows)
                   .sort_values(['f1', 'section'], na_position='first')
                   .reset_index(drop=True))
section_summary.insert(0, 'attention_rank', np.arange(1, len(section_summary) + 1))
display(section_summary[['attention_rank', 'section', 'tp', 'fp', 'fn',
                         'precision', 'recall', 'f1']].round(4))

weakest = section_summary.iloc[0]
print(f"Needs attention first: {weakest['section']} (F1={weakest['f1']:.4f})")
section_chart = section_summary.melt(
    id_vars=['section', 'attention_rank'],
    value_vars=['precision', 'recall', 'f1'],
    var_name='metric', value_name='score',
)
figure = px.bar(
    section_chart, x='section', y='score', color='metric', barmode='group',
    category_orders={'section': section_summary.section.tolist(),
                     'metric': ['precision', 'recall', 'f1']},
    range_y=[0, 1], title='Section scores: latest selected experiment',
    hover_data={'score': ':.4f', 'attention_rank': True},
)
figure.add_hline(
    y=TARGET_F1, line_dash='dash', line_color='red', line_width=2,
    annotation_text=f'Target F1 = {TARGET_F1:.0%}',
    annotation_position='top left',
)
figure.update_layout(xaxis_title='Section (lowest F1 first)',
                     yaxis_title='Exact-match score')
figure.show()

## Contact

In [ ]:
contact_summary, contact_errors = show_section('Contact', ['contact.name', 'contact.location'])

## Skills — primary reported metric

The table below is the report metric: pooled item-level TP/FP/FN against the current manually reviewed gold for the selected experiment.

In [ ]:
skills_summary, skills_errors = show_section('Skills', ['skills'])

### Skills section text similarity

This practical diagnostic compares each predicted and gold Skills list as one normalized text block using local TF-IDF cosine similarity. It tolerates list-boundary differences such as a combined `Web Intelligence, Crystal and Dashboard Reports` phrase. Section presence coverage and strict entity F1 remain visible separately.


In [ ]:
skills_text_rows = pd.DataFrame(skills_text_scoring.score_skills_records(selected.to_dict('records')))
skills_text_applicable = skills_text_rows[skills_text_rows.skills_applicable.eq(1)].copy()
if skills_text_applicable.empty:
    print('No resumes with gold skills in the selected experiment.')
else:
    skills_presence_coverage = skills_text_applicable.skills_section_found.mean()
    skills_mean_cosine = skills_text_applicable.skills_text_cosine.mean()
    skills_median_cosine = skills_text_applicable.skills_text_cosine.median()
    display(pd.DataFrame([
        {'metric': 'Skills sections in gold', 'value': len(skills_text_applicable)},
        {'metric': 'Skills section presence coverage', 'value': skills_presence_coverage},
        {'metric': 'Mean skills TF-IDF cosine', 'value': skills_mean_cosine},
        {'metric': 'Median skills TF-IDF cosine', 'value': skills_median_cosine},
    ]).round(4))
    skills_chart = pd.DataFrame({
        'metric': ['Section presence coverage', 'Mean text similarity'],
        'score': [skills_presence_coverage, skills_mean_cosine],
    })
    figure = px.bar(skills_chart, x='metric', y='score', text='score',
                    range_y=[0, 1], title='Skills extraction: practical section-level scores')
    figure.update_traces(texttemplate='%{text:.1%}', textposition='outside')
    figure.update_layout(xaxis_title=None, yaxis_title='Score')
    figure.show()
    display(skills_text_applicable.nsmallest(10, 'skills_text_cosine')[[
        'resume_id', 'skills_text_cosine', 'predicted_skills_text', 'gold_skills_text'
    ]].round(4))


### Legacy original-annotation diagnostic (disabled)

This diagnostic is intentionally disabled because the report uses the current manually reviewed gold and pooled field-level TP/FP/FN. It must not be interpreted as the reported Skills score.

In [ ]:
print('Legacy original-gold Skills diagnostic disabled; use the primary pooled Skills table above.')

## Education

In [ ]:
education_summary, education_errors = show_section('Education', ['education.degree', 'education.field', 'education.institution', 'education.start_year', 'education.end_year'])

## Experience

In [ ]:
experience_summary, experience_errors = show_section('Experience', ['experience.job_title', 'experience.company', 'experience.location', 'experience.start_date', 'experience.end_date', 'experience.current_role'])

## Projects

In [ ]:
projects_summary, projects_errors = show_section('Projects', ['projects.name', 'projects.technologies'])

## Certifications

In [ ]:
certifications_summary, certifications_errors = show_section('Certifications', ['certifications.name', 'certifications.issuer', 'certifications.year'])

## Description diagnostics

Each gold experience is compared with the predicted experience at the same list index because both profiles preserve resume reading order. Title, company and date agreement is shown only as an ordering diagnostic and does not block description scoring. TF-IDF cosine similarity is calculated without an embedding or generative model. A missing positional experience or description scores zero, while gold experiences without descriptions are not applicable. This diagnostic is reported separately from field TP/FP/FN because the historical gold descriptions are not guaranteed verbatim transcriptions.

In [ ]:
description_records = selected.to_dict('records')
description_rows = pd.DataFrame(description_scoring.score_description_records(description_records))
description_entries = pd.DataFrame(description_scoring.score_all_description_entries(description_records))
if description_entries.empty:
    print('No applicable gold experience descriptions in this experiment.')
else:
    coverage = description_entries.description_found.mean()
    cosine = description_entries.description_cosine.mean()
    matched = description_entries[description_entries.description_found.eq(1)]
    found_similarity = matched.description_cosine.mean() if len(matched) else 0.0
    print(f'{len(description_entries)} gold jobs had descriptions; {len(matched)} same-index predicted jobs had descriptions.')
    print(f'Positional description coverage: {coverage:.1%} | Similarity where present: {found_similarity:.1%} | Overall with missing positions scored 0: {cosine:.1%}')
    summary = pd.DataFrame({
        'measure': ['Same-index descriptions present', 'Similarity where present', 'Overall (missing positions = 0)'],
        'score': [coverage, found_similarity, cosine],
    })
    description_figure = px.bar(summary, x='measure', y='score', text='score', range_y=[0, 1], title='Experience description results')
    description_figure.update_traces(texttemplate='%{text:.1%}', textposition='outside')
    description_figure.update_layout(xaxis_title=None, yaxis_title='Score', showlegend=False)
    description_figure.show()
    display(description_entries.nsmallest(10, 'description_cosine')[[
        'resume_id', 'gold_job_title', 'gold_company', 'gold_start_date', 'gold_end_date',
        'predicted_job_title', 'predicted_company', 'description_found',
        'identity_match_score', 'description_cosine', 'predicted_description', 'gold_description',
    ]].round(4))

## Gemma versus Gemini average latency

This comparison uses only resume IDs completed by both models and selects the most recent raw inference experiment for each model. Offline rescoring rows are excluded.

In [ ]:
comparison_models = ['gemma-4-26b-a4b-it-maas', 'gemini-3.5-flash']
raw_success = ledger[ledger.status.eq('success')].copy()
if 'evaluation_type' in raw_success.columns:
    raw_success = raw_success[~raw_success.evaluation_type.eq('offline_normalized_scoring')]
model_runs = {}
for model_name in comparison_models:
    candidates = raw_success[raw_success.model.eq(model_name)].copy()
    if candidates.empty:
        continue
    latest_key = candidates.sort_values('timestamp_utc').iloc[-1].experiment_key
    model_runs[model_name] = (candidates[candidates.experiment_key.eq(latest_key)]
                              .sort_values('timestamp_utc')
                              .drop_duplicates('resume_id', keep='last'))
if len(model_runs) < 2:
    print('Run both Gemma and Gemini experiments before comparing latency.')
else:
    paired_ids = set.intersection(*(set(frame.resume_id) for frame in model_runs.values()))
    latency_comparison = pd.DataFrame([
        {
            'model': model_name,
            'paired_resumes': len(paired_ids),
            'average_resume_seconds': frame[frame.resume_id.isin(paired_ids)].resume_total_seconds.mean(),
        }
        for model_name, frame in model_runs.items()
    ])
    display(latency_comparison.round(3))
    latency_comparison_figure = px.bar(
        latency_comparison, x='model', y='average_resume_seconds',
        text='average_resume_seconds', title=f'Average latency on {len(paired_ids)} paired resumes',
    )
    latency_comparison_figure.update_traces(texttemplate='%{text:.2f}s', textposition='outside')
    latency_comparison_figure.update_layout(xaxis_title=None, yaxis_title='Average seconds per resume', showlegend=False)
    latency_comparison_figure.show()

## Latency

In [ ]:
latency = selected[['resume_id', 'resume_total_seconds', 'model_wall_seconds', 'page_count']].copy()
display(latency.describe(include='all').round(3))
latency.sort_values('resume_total_seconds').plot.bar(x='resume_id', y='resume_total_seconds',
    figsize=(max(9, len(latency) * .25), 4), legend=False, title='Total latency per resume')
plt.ylabel('Seconds'); plt.xticks([]); plt.grid(axis='y', alpha=.25); plt.tight_layout(); plt.show()

## Token usage and estimated cost

Provider-reported tokens are available only for runs created after usage capture was introduced. Cost is an estimate using the pricing version stored on each attempt; Cloud Billing remains authoritative.

In [ ]:
usage_columns = ['prompt_tokens', 'output_tokens', 'total_tokens', 'cached_input_tokens', 'estimated_cost_usd']
for column in usage_columns:
    if column not in selected.columns:
        selected[column] = np.nan
usage = selected[['resume_id', 'page_count', *usage_columns]].copy()
measured_usage = usage[usage.total_tokens.notna()].copy()
print({'selected_resumes': len(usage), 'resumes_with_usage_metadata': len(measured_usage),
       'coverage': round(len(measured_usage) / len(usage), 4) if len(usage) else 0})
if measured_usage.empty:
    print('No stored token metadata for this experiment; rerun with the updated provider.')
else:
    display(measured_usage.describe().round(6))
    print({'total_prompt_tokens': int(measured_usage.prompt_tokens.sum()),
           'total_output_tokens': int(measured_usage.output_tokens.sum()),
           'total_tokens': int(measured_usage.total_tokens.sum()),
           'estimated_total_cost_usd': round(measured_usage.estimated_cost_usd.sum(), 6),
           'mean_cost_per_resume_usd': round(measured_usage.estimated_cost_usd.mean(), 6),
           'resumes_per_minute': round(60 / measured_usage.merge(latency, on='resume_id').resume_total_seconds.mean(), 3)})
    display(measured_usage.sort_values('estimated_cost_usd', ascending=False).head(20))

## Metrics across experiment versions

In [ ]:
latest_per_experiment = (success_log.sort_values('timestamp_utc')
    .drop_duplicates(['experiment_key', 'resume_id'], keep='last'))
trend_rows = []
for key, group in latest_per_experiment.groupby('experiment_key', sort=False):
    experiment_counts = []
    for record in group.to_dict('records'):
        for result in record.get('field_counts', []):
            experiment_counts.append({'resume_id': record['resume_id'], **result})
    if not experiment_counts:
        continue
    experiment_summary = summarize(pd.DataFrame(experiment_counts))
    first = group.iloc[0]
    for metric in experiment_summary.to_dict('records'):
        trend_rows.append({
            'experiment_key': key, 'prompt_version': first.get('prompt_version'),
            'model': first.get('model'), 'evaluation_mode': first.get('evaluation_mode'),
            'first_timestamp_utc': group.timestamp_utc.min(),
            'resumes': group.resume_id.nunique(), **metric,
        })
trends = pd.DataFrame(trend_rows).sort_values(['first_timestamp_utc', 'field'])
if trends.experiment_key.nunique() < 2:
    print('One experiment is currently available; trend lines will appear after another experiment is logged.')
display(trends[['prompt_version', 'model', 'resumes', 'field', 'precision', 'recall', 'f1']].round(4))
TREND_FIELD = 'contact.name'  # Change to one field from sorted(trends.field.unique())
trend_view = trends[trends.field.eq(TREND_FIELD)].sort_values('first_timestamp_utc').copy()
trend_view['experiment'] = trend_view.apply(
    lambda row: f"{row['prompt_version']} | {row.get('evaluation_mode', '')}", axis=1
)
trend_long = trend_view.melt(
    id_vars=['experiment', 'resumes'], value_vars=['precision', 'recall', 'f1'],
    var_name='metric', value_name='score',
)
trend_figure = px.line(
    trend_long, x='experiment', y='score', color='metric', markers=True,
    range_y=[0, 1], title=f'Experiment history: {TREND_FIELD}',
    hover_data=['resumes'],
)
trend_figure.update_layout(xaxis_title='Experiment run', yaxis_title='Score')
trend_figure.show()

## Best Gemma result against the current gold

This comparison recalculates both models against the effective gold loaded in this notebook. The strongest near-complete Gemma run contains 83 of 86 resumes, so its coverage is shown explicitly and it is not presented as a complete acceptance run.

In [ ]:
BEST_GEMMA_EXPERIMENT_KEY = 'gemma-4-26b-a4b-it-maas:v006_atomic_skills_project_certification:e732b80515f0285e3cd2e3f6efbb5f4ea3bb8dc1a70480fe38a2c60b7fcac2fc:temp=0.0:dpi=150:evaluation=production_output_atomic_skills_project_certification_usage_v010:normalize_for_scoring=False:location_policy=locality_only_v001:provider=36927c81ecef4ab69102842615de9d93f113ba43283206abac9b9eaf8ced127b:gold=education_source_review_v001'
REPORT_MODEL_KEYS = {
    'Gemma 4': BEST_GEMMA_EXPERIMENT_KEY,
    'Gemini 3.5 Flash v10 (86/86)': NORMALIZATION_SOURCE_EXPERIMENT_KEY,
}
REPORT_SECTIONS = {
    'Contact': ('contact.',), 'Skills': ('skills',),
    'Education': ('education.',), 'Experience': ('experience.',),
    'Projects': ('projects.',), 'Certifications': ('certifications.',),
}

current_gold = {example.resume_id: example.profile for example in gold.load(None)}
ledger_rows = read_jsonl(LEDGER_PATH)
comparison_rows = []
for model_label, source_key in REPORT_MODEL_KEYS.items():
    latest = {}
    for row in sorted(ledger_rows, key=lambda item: item.get('timestamp_utc', '')):
        if (row.get('status') == 'success' and row.get('experiment_key') == source_key
                and row.get('evaluation_type') != 'offline_normalized_scoring'):
            latest[row['resume_id']] = row
    evidence = []
    for resume_id, row in latest.items():
        if resume_id not in current_gold:
            continue
        prediction = row.get('production_prediction') or row.get('prediction') or {}
        evidence.extend(field_scoring.confusion_rows(
            prediction, current_gold[resume_id], mode='normalized'
        ))
    for section, prefixes in REPORT_SECTIONS.items():
        section_rows = [item for item in evidence if any(
            item['field'] == prefix or item['field'].startswith(prefix)
            for prefix in prefixes
        )]
        tp = sum(item['tp'] for item in section_rows)
        fp = sum(item['fp'] for item in section_rows)
        fn = sum(item['fn'] for item in section_rows)
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
        comparison_rows.append({
            'model': model_label, 'resumes': len(latest), 'section': section,
            'tp': tp, 'fp': fp, 'fn': fn, 'precision': precision,
            'recall': recall, 'f1': f1, 'passes_075': bool(f1 >= TARGET_F1),
        })

model_comparison = pd.DataFrame(comparison_rows)
display(model_comparison.round(4))
comparison_figure = px.bar(
    model_comparison, x='section', y='f1', color='model', barmode='group',
    category_orders={'section': list(REPORT_SECTIONS)}, range_y=[0, 1],
    text=model_comparison.f1.map(lambda value: f'{value:.3f}'),
    hover_data=['resumes', 'precision', 'recall', 'tp', 'fp', 'fn'],
    title='Resume parsing F1 by section — rescored against current gold',
)
comparison_figure.add_hline(
    y=TARGET_F1, line_dash='dash', line_color='red',
    annotation_text=f'Target F1 = {TARGET_F1:.0%}', annotation_position='top left',
)
comparison_figure.update_traces(textposition='outside')
comparison_figure.update_layout(xaxis_title='Section', yaxis_title='Normalized micro-F1')
comparison_figure.show()
model_comparison.to_csv(EXPORT_DIR / 'gemma_gemini_current_gold_comparison.csv', index=False)

## Searchable evidence and exports

In [ ]:
def find_errors(resume_id=None, field=None, error='any'):
    result = counts.copy()
    if resume_id is not None:
        result = result[result.resume_id.eq(resume_id)]
    if field is not None:
        result = result[result.field.eq(field)]
    if error == 'fp': result = result[result.fp.gt(0)]
    elif error == 'fn': result = result[result.fn.gt(0)]
    elif error == 'any': result = result[result.fp.gt(0) | result.fn.gt(0)]
    elif error != 'all': raise ValueError("error must be 'fp', 'fn', 'any', or 'all'")
    return result.sort_values(['field', 'resume_id'])

summary.to_csv(EXPORT_DIR / 'selected_experiment_field_summary.csv', index=False)
counts.to_json(EXPORT_DIR / 'selected_experiment_evidence.jsonl', orient='records', lines=True)
latency.to_csv(EXPORT_DIR / 'selected_experiment_latency.csv', index=False)
trends.to_csv(EXPORT_DIR / 'experiment_metric_trends.csv', index=False)
print('Exports:', EXPORT_DIR)
SIDE_BY_SIDE_DIR = LEDGER_PATH.parent / 'side_by_side_review'
for experiment_key, group in latest_per_experiment.groupby('experiment_key', sort=False):
    first = group.iloc[0]
    folder_name = str(first.get('evaluation_mode') or first.get('prompt_version') or 'experiment')
    experiment_dir = SIDE_BY_SIDE_DIR / folder_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    for record in group.to_dict('records'):
        resume_id = record['resume_id']
        (experiment_dir / f'{resume_id}__gold.json').write_text(
            json.dumps(record['reference'], ensure_ascii=False, indent=2), encoding='utf-8'
        )
        (experiment_dir / f'{resume_id}__raw_predicted.json').write_text(
            json.dumps(record['raw_prediction'], ensure_ascii=False, indent=2), encoding='utf-8'
        )
print('Side-by-side experiment folders:', SIDE_BY_SIDE_DIR)
display(find_errors(error='any').head(25))